# 18 — The guarded sweep (E15 completion; director package spec step 0; VERSION switch added v0.17)

**VERSION = "v3"** (cell 1) runs the per-block-floor sweeps on the curated-EFG-block re-solve (`runs_v3/`, 12 design formulations, after 12 has produced their anchors); `"v1"` = the as-frozen record (`runs/`, done 2026-09-04). Measured v1 cost: 52–76 min per formulation under contention.

Spec v0.14.1 authorizes extending the per-block-floor MGA sweeps from {S0, S4} (notebook 16)
to **all 14 frozen formulations**, plus the queued **guarded MAA spot-check on S0**. Guarded
semantics = the applied headline ("no value theme left more than 5% behind"); the aggregate
band remains the Claim-A estimand.

Per formulation, into `analyses/y2y/runs/<formulation_id>/`:

| artifact | what | config |
|---|---|---|
| `mga_guard_g05.tif` + `certificates_guard.csv` | 50 guarded MGA members | band g=5% + 4 block floors capture_b ≥ 0.95·anchor_b |
| `guard_meta.json` | re-solved anchor vs the frozen record, timings | — |
| S0 only: `maa_guard_g05.tif` + `certificates_maa_guard.csv` | guarded MAA spot-check | E12 seed 20260902 + the same floors |

**Anchors, k-best pools and twins STAND** — floors are anchor-relative, so the anchor is
re-solved only to obtain the compiled model + `x` (asserted ≤1e-3 relative from the frozen
`formulation_meta.json` objective). Byte-compatible with 16's S0/S4 outputs (skipped here).
Measured cost 11–15 min per formulation (floors *shrink* the search) ⇒ **~3 h serial for the
12 open formulations** — the spec's "8–10 h" is superseded by this measurement. Fully
resumable; **live internet throughout** (WLS). Kernel `R (y2y)`.


In [11]:
ANALYSIS <- "y2y"
PROJ <- normalizePath(getwd())
while (!file.exists(file.path(PROJ, "config.py"))) {
  parent <- dirname(PROJ)
  if (identical(parent, PROJ)) stop("config.py not found above getwd() -- open from inside the repo")
  PROJ <- parent
}
setwd(PROJ)
source(file.path(PROJ, "prioritizr_core.R"))
source(file.path(PROJ, "mga_core.R"))

mpath <- pr_refresh_manifest(PROJ, ANALYSIS)

# ---- VERSION switch (study plan v0.17; supersede, never delete) --------------------------------------------
# "v1" = the as-frozen 2026-08-30 record (spec/manifest.csv, runs/); "v3" = the curated EFG block re-solve
# (spec/manifest_v3.csv, runs_v3/; 12 design formulations). config.EFG_SUBDIR must match (asserted after ingest).
VERSION <- "v3.1"      # v3.1 = the curated block with window-derived targets (study plan v0.17.3)
MANIFEST_REL <- if (VERSION == "v1") "analyses/y2y/spec/manifest.csv" else sprintf("analyses/y2y/spec/manifest_%s.csv", VERSION)
FREEZE_REL   <- if (VERSION == "v1") "analyses/y2y/spec/manifest_freeze.sha256" else sprintf("analyses/y2y/spec/manifest_%s.sha256", VERSION)
RUNS_REL     <- if (VERSION == "v1") "analyses/y2y/runs" else sprintf("analyses/y2y/runs_%s", VERSION)
EFG_SUBDIR_EXPECTED <- if (VERSION == "v1") "iucn_efg" else paste0("iucn_efg_", sub("\\..*$", "", VERSION))   # minor versions share the block
MAN <- read.csv(file.path(PROJ, MANIFEST_REL), stringsAsFactors = FALSE)
for (col in c("kbest_ref", "twin_ref")) {           # v3 has no pool/twin pointers: an all-empty column reads as NA
  if (!col %in% names(MAN)) MAN[[col]] <- ""
  MAN[[col]] <- ifelse(is.na(MAN[[col]]), "", as.character(MAN[[col]]))
}
stopifnot(nrow(MAN) %in% c(12, 14))
dig <- strsplit(readLines(file.path(PROJ, FREEZE_REL))[1], "  ")[[1]][1]
stopifnot("manifest does not match its freeze hash -- STOP" =
            identical(unname(tools::sha256sum(file.path(PROJ, MANIFEST_REL))[[1]]), dig))
cat(sprintf("VERSION %s: %s verified against freeze hash %s... (%d formulations)\n", VERSION, basename(MANIFEST_REL), substr(dig, 1, 16), nrow(MAN)))
RUNS <- file.path(PROJ, RUNS_REL)
REAL245 <- "input_data/aligned_stack/climate_realizations/macrorefugia_245_2071_2100.tif"
ER <- jsonlite::read_json(file.path(PROJ, "analyses/y2y/spec/e_round_v13.json"))
BLOCKS <- lapply(ER$e17_t3$blocks, unlist)       # == config.BLOCKS (4 PROACT blocks)
stopifnot(setequal(names(BLOCKS), c("core_habitat", "connectivity", "carbon", "biodiversity")))
FLOOR_G <- 0.05                                   # capture_b >= (1 - FLOOR_G) * anchor_b
MAA_SPOTCHECK <- "s0_ssp585_theta5"
cat("floors on blocks:", paste(names(BLOCKS), collapse = ", "), "| floor g =", FLOOR_G, "\n")


manifest refreshed from config.py (analysis=y2y)
VERSION v3.1: manifest_v3.1.csv verified against freeze hash 259f35eddcc33476... (12 formulations)
floors on blocks: core_habitat, connectivity, carbon, biodiversity | floor g = 0.05 


In [12]:
# ---- two ingested base contexts, built ONCE (one per climate level) ------------------------
ctx585 <- pr_setup(mpath, PROJ)
ctx585 <- modifyList(ctx585, pr_ingest(ctx585))
ctx585 <- modifyList(ctx585, pr_planning_units(ctx585))

ctx245 <- pr_setup(mpath, PROJ)
ctx245$layers$path[ctx245$layers$name == "climate_type_macrorefugia"] <- REAL245
ctx245 <- modifyList(ctx245, pr_ingest(ctx245))
ctx245 <- modifyList(ctx245, pr_planning_units(ctx245))
efg_used <- ctx585$layers$path[ctx585$layers$role == "feature_efg"]
stopifnot("manifest.json enumerates a different EFG block than VERSION expects -- check config.EFG_SUBDIR" =
            length(efg_used) > 0 && all(grepl(paste0("/", EFG_SUBDIR_EXPECTED, "/"), efg_used)))
cat(sprintf("EFG block: %d features from %s/\n", length(efg_used), EFG_SUBDIR_EXPECTED))
cat("base contexts ready (585 canonical; 245 with the realization layer patched)\n")

base_for <- function(row) if (grepl("^ssp245", row$climate_level)) ctx245 else ctx585
form_wt  <- function(row) list(w = jsonlite::fromJSON(row$weight_vector),
                               t = jsonlite::fromJSON(row$target_vector))


prioritizr 8.1.0 | terra 1.9.34 | analysis=y2y | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=full | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=0
outputs -> output_data/iter6_y2y
ingested 28 features (8 continuous + 20 EFG) + cost + PA mask | grid 1286 x 3312 @ 1000 m
normalized 28 features to total=100000 each (scale-invariant conditioning)
planning units: 1,272,914 cells | budget = 30% = 381,874 cells
locked-in [pa_mask]: 191,029 cells (15.0% of window) -- fits within budget
prioritizr 8.1.0 | terra 1.9.34 | analysis=y2y | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=full | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=0
outputs -> output_data/iter6_y2y
ingested 28 feature

In [13]:
# ---- DRY PLAN (no solves): what this run will do -------------------------------------------
for (i in seq_len(nrow(MAN))) {
  row <- MAN[i, ]; cd <- file.path(RUNS, row$formulation_id)
  st <- if (file.exists(file.path(cd, "mga_guard_g05.tif"))) "done" else "TODO"
  maa <- if (row$formulation_id == MAA_SPOTCHECK)
           (if (file.exists(file.path(cd, "maa_guard_g05.tif"))) " | maa-guard: done" else " | maa-guard: TODO") else ""
  cat(sprintf("%-22s %-9s mga-guard: %-5s%s\n", row$formulation_id,
              sub("_2071_2100", "", row$climate_level), st, maa))
}


s0_ssp585_theta5       ssp585    mga-guard: done  | maa-guard: TODO
s1_ssp585_theta5       ssp585    mga-guard: done 
s2_ssp585_theta5       ssp585    mga-guard: done 
s3_ssp585_theta5       ssp585    mga-guard: done 
s4_ssp585_theta3       ssp585    mga-guard: done 
s5_ssp585_theta5       ssp585    mga-guard: done 
s0_ssp245_theta5       ssp245    mga-guard: done 
s1_ssp245_theta5       ssp245    mga-guard: done 
s2_ssp245_theta5       ssp245    mga-guard: TODO 
s3_ssp245_theta5       ssp245    mga-guard: TODO 
s4_ssp245_theta3       ssp245    mga-guard: TODO 
s5_ssp245_theta5       ssp245    mga-guard: TODO 


In [14]:
# ---- the guarded-sweep runner (one artifact; skipped when its tif exists) -------------------
run_guard <- function(row, kind = c("mga", "maa")) {
  kind <- match.arg(kind)
  cd <- file.path(RUNS, row$formulation_id)
  tif <- file.path(cd, if (kind == "mga") "mga_guard_g05.tif" else "maa_guard_g05.tif")
  csv <- file.path(cd, if (kind == "mga") "certificates_guard.csv" else "certificates_maa_guard.csv")
  if (file.exists(tif)) { cat(sprintf("   %s/%s-guard exists -- skipped\n", row$formulation_id, kind)); return(invisible(NULL)) }
  stopifnot(file.exists(file.path(cd, "formulation_meta.json")), file.exists(file.path(cd, "anchor.tif")))
  meta <- jsonlite::read_json(file.path(cd, "formulation_meta.json"))
  wt <- form_wt(row)
  actx <- pr_override(base_for(row),
      targets = wt$t, feature_weight_multipliers = wt$w,
      results_dir = file.path(RUNS_REL, row$formulation_id),
      results_subdir = "guard_build",
      solver = "gurobi", decision_type = "binary", opt_gap = 1e-4, portfolio_n = 1)
  actx <- modifyList(actx, pr_weights(actx))
  actx <- modifyList(actx, pr_targets(actx))
  actx <- modifyList(actx, pr_penalty_matrices(actx))
  bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
  cm <- mga_compile(actx)
  t0 <- proc.time()[["elapsed"]]
  anchor <- mga_anchor(cm, opt_gap = row$opt_gap)
  # the frozen record STANDS: the re-solve only supplies the compiled model + x for the floors
  rel <- abs(anchor$z - meta$anchor_objective) / abs(meta$anchor_objective)
  stopifnot("re-solved anchor drifted > 1e-3 relative from formulation_meta.json -- STOP" = rel <= 1e-3)
  a_frozen <- terra::values(terra::rast(file.path(cd, "anchor.tif")))[cm$pu_index] == 1
  ham_frozen <- sum(a_frozen != anchor$x)
  cat(sprintf("   anchor re-solved %.6f vs frozen %.6f (rel %.1e) | %d cells differ from anchor.tif (near-ties)\n",
              anchor$z, meta$anchor_objective, rel, ham_frozen))
  floors <- list(ctx = actx, blocks = BLOCKS, g = FLOOR_G)
  gen <- if (kind == "mga") {
    mga_generate(cm, anchor, g = row$band_gap_g, k = row$k_requested, floors = floors)
  } else {
    seed <- ER$e12$seeds[[row$formulation_id]]
    stopifnot("no E12 seed recorded for this formulation" = !is.null(seed))
    maa_generate(cm, anchor, g = row$band_gap_g, k = row$k_requested, seed = seed, floors = floors)
  }
  layers <- lapply(seq_len(gen$k), function(i) {
    r <- terra::rast(actx$cost); v <- rep(NA_integer_, terra::ncell(r))
    v[cm$pu_index] <- as.integer(gen$members[i, ]); terra::values(r) <- v; r })
  s <- terra::rast(layers)
  names(s) <- sprintf(if (kind == "mga") "guard_%02d" else "maag_%02d", seq_len(gen$k))
  terra::writeRaster(s, tif, overwrite = TRUE, datatype = "INT1U", NAflag = 255,
                     gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
  write.csv(gen$certificates, csv, row.names = FALSE)
  gm <- file.path(cd, if (kind == "mga") "guard_meta.json" else "maa_guard_meta.json")
  jsonlite::write_json(list(
    formulation_id = row$formulation_id, kind = kind, floor_g = FLOOR_G, blocks = BLOCKS,
    band_g = row$band_gap_g, k = gen$k, anchor_objective_resolved = anchor$z,
    anchor_objective_frozen = meta$anchor_objective, anchor_rel_drift = rel,
    anchor_cells_differing = ham_frozen, seed = if (kind == "maa") gen$seed else NULL,
    total_runtime_s = sum(gen$certificates$runtime_s),
    wall_s = proc.time()[["elapsed"]] - t0, created_utc = format(Sys.time(), tz = "UTC")),
    gm, auto_unbox = TRUE, pretty = TRUE, digits = 10)
  cat(sprintf("   wrote %s (+ certificates, meta)\n", basename(tif)))
  invisible(NULL)
}


In [ ]:
# ---- THE LOOP: serial over the frozen formulations, then the S0 MAA spot-check --------------
t_batch <- proc.time()[["elapsed"]]
for (i in seq_len(nrow(MAN))) {
  row <- MAN[i, ]
  cat(sprintf("\n===================== %s (%d/%d) =====================\n", row$formulation_id, i, nrow(MAN)))
  run_guard(row, "mga")
  cat(sprintf("== %s | batch elapsed %.2f h\n", row$formulation_id, (proc.time()[["elapsed"]] - t_batch) / 3600))
}
cat(sprintf("\n===================== guarded MAA spot-check: %s =====================\n", MAA_SPOTCHECK))
run_guard(MAN[MAN$formulation_id == MAA_SPOTCHECK, ], "maa")
cat("\nGUARDED SWEEP COMPLETE -- next: analyses/y2y/19_director_surfaces.ipynb\n")



===================== s0_ssp585_theta5 (1/12) =====================
   s0_ssp585_theta5/mga-guard exists -- skipped
== s0_ssp585_theta5 | batch elapsed 0.00 h

===================== s1_ssp585_theta5 (2/12) =====================
   s1_ssp585_theta5/mga-guard exists -- skipped
== s1_ssp585_theta5 | batch elapsed 0.00 h

===================== s2_ssp585_theta5 (3/12) =====================
   s2_ssp585_theta5/mga-guard exists -- skipped
== s2_ssp585_theta5 | batch elapsed 0.00 h

===================== s3_ssp585_theta5 (4/12) =====================
   s3_ssp585_theta5/mga-guard exists -- skipped
== s3_ssp585_theta5 | batch elapsed 0.00 h

===================== s4_ssp585_theta3 (5/12) =====================
   s4_ssp585_theta3/mga-guard exists -- skipped
== s4_ssp585_theta3 | batch elapsed 0.00 h

===================== s5_ssp585_theta5 (6/12) =====================
   s5_ssp585_theta5/mga-guard exists -- skipped
== s5_ssp585_theta5 | batch elapsed 0.00 h

===================== s0_ssp245_theta5 

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (28 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.1 and 1)
││└•weights:    continuous values (between 0.05 and 2.34271)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



compiled: 1272942 cols (1272914 pu + 28 aux) x 29 rows | 191029 locked pu | modelsense min
anchor: objective 5.056016 (bound 5.055956, gap 1.19e-05) | 381,874 selected | 683 s
   anchor re-solved 5.056016 vs frozen 5.056016 (rel 4.4e-12) | 0 cells differ from anchor.tif (near-ties)
band wall appended: obj0 . x <= 5.308817  (g = 0.05 on z* = 5.056016)
  block floor core_habitat   anchor capture 0.4518 -> floor 0.4292
  block floor connectivity   anchor capture 0.7229 -> floor 0.6868
  block floor carbon         anchor capture 0.6516 -> floor 0.6190
  block floor biodiversity   anchor capture 0.6147 -> floor 0.5840
g=0.05 iter 01/50: band 5.308706 (+5.00% of z*) OK | ham(anchor) 303,354 | 58 s
g=0.05 iter 02/50: band 5.308818 (+5.00% of z*) OK | ham(anchor) 200,524 | 63 s
g=0.05 iter 03/50: band 5.308727 (+5.00% of z*) OK | ham(anchor) 187,052 | 61 s
g=0.05 iter 04/50: band 5.308817 (+5.00% of z*) OK | ham(anchor) 188,796 | 58 s
g=0.05 iter 05/50: band 5.301061 (+4.85% of z*) OK | ham(an

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (28 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.1 and 1)
││└•weights:    continuous values (between 0.05 and 2.781598)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



In [ ]:
# ---- integrity + cost summary (results_log-ready) -------------------------------------------
tot <- 0
for (i in seq_len(nrow(MAN))) {
  row <- MAN[i, ]; cd <- file.path(RUNS, row$formulation_id)
  csv <- file.path(cd, "certificates_guard.csv")
  if (!file.exists(csv)) { cat(sprintf("%-22s MISSING\n", row$formulation_id)); next }
  ce <- read.csv(csv)
  gm <- file.path(cd, "guard_meta.json")
  drift <- if (file.exists(gm)) jsonlite::read_json(gm)$anchor_rel_drift else NA
  tot <- tot + sum(ce$runtime_s)
  cat(sprintf("%-22s %2d members | band_ok %s | dup %d | time-limited %d | %5.1f min | anchor drift %s\n",
              row$formulation_id, nrow(ce), if (all(ce$band_ok)) "ALL" else "VIOLATED",
              sum(ce$duplicate), sum(ce$status == "TIME_LIMIT"), sum(ce$runtime_s) / 60,
              if (is.na(drift)) "(16-era)" else sprintf("%.1e", drift)))
}
cat(sprintf("\nsweep solve time (all guarded MGA members): %.1f h\n", tot / 3600))
mcsv <- file.path(RUNS, MAA_SPOTCHECK, "certificates_maa_guard.csv")
if (file.exists(mcsv)) {
  ce <- read.csv(mcsv)
  cat(sprintf("%s guarded MAA: %d members | band_ok %s | dup %d | %.1f min | seed %d\n", MAA_SPOTCHECK,
              nrow(ce), if (all(ce$band_ok)) "ALL" else "VIOLATED", sum(ce$duplicate),
              sum(ce$runtime_s) / 60, ce$seed[1]))
}


s0_ssp585_theta5       50 members | band_ok ALL | dup 0 | time-limited 0 |  11.2 min | anchor drift (16-era)
s1_ssp585_theta5       50 members | band_ok ALL | dup 0 | time-limited 0 |  55.6 min | anchor drift 1.3e-06
s2_ssp585_theta5       50 members | band_ok ALL | dup 0 | time-limited 0 |  61.9 min | anchor drift 6.6e-06
s3_ssp585_theta5       50 members | band_ok ALL | dup 0 | time-limited 0 |  60.4 min | anchor drift 1.4e-06
s4_ssp585_theta3       50 members | band_ok ALL | dup 0 | time-limited 0 |  12.0 min | anchor drift (16-era)
s5_ssp585_theta5       50 members | band_ok ALL | dup 0 | time-limited 0 |  75.6 min | anchor drift 3.7e-06
s0_ssp245_theta5       50 members | band_ok ALL | dup 0 | time-limited 0 |  70.6 min | anchor drift 2.1e-06
s1_ssp245_theta5       50 members | band_ok ALL | dup 0 | time-limited 0 |  52.5 min | anchor drift 7.7e-06
s2_ssp245_theta5       50 members | band_ok ALL | dup 0 | time-limited 0 |  57.1 min | anchor drift 8.8e-06
s3_ssp245_theta5       50 